In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os
D="/content/drive/MyDrive/phd_data/"
print([f for f in os.listdir(D) if "KDD" in f.upper()])

In [ ]:
tr=pd.read_csv(D+"KDDTrain+.txt",header=None,sep="\t")
te=pd.read_csv(D+"KDDTest+.txt",header=None,sep="\t")
print(tr.shape, te.shape)
print(tr.iloc[:,41].nunique(), te.iloc[:,41].nunique())
print(te.iloc[:,41].value_counts().head(5))

In [ ]:
print(open(D+"KDDTrain+.txt").readline()[:300])

In [ ]:
import pandas as pd, os
D="/content/drive/MyDrive/phd_data/"
print([f for f in os.listdir(D) if "KDD" in f])
te=pd.read_csv(D+"KDDTest_Labelled.txt",header=None)
tk=pd.read_csv(D+"KDDTrain_kaggle.txt",header=None)
tz=pd.read_csv(D+"KDDTrain+.txt",header=None,sep="\t")
print(te.shape, tk.shape, tz.shape)
print(te.iloc[:,41].nunique(), tk.iloc[:,41].nunique())
print(te.iloc[:,41].value_counts().head(8))

In [ ]:
import pandas as pd
D="/content/drive/MyDrive/phd_data/"
def rd(f):
    s="\t" if "\t" in open(D+f).readline() else ","
    return pd.read_csv(D+f,header=None,sep=s)
te=rd("KDDTest_Labelled.txt")
tr=rd("KDDTrain+.txt")
t2=rd("KDDTest+.txt")
print("labelled test", te.shape, te.iloc[:,41].nunique())
print("train", tr.shape, tr.iloc[:,41].nunique())
print("KDDTest+.txt", t2.shape, t2.iloc[:,41].nunique())
print(te.iloc[:,41].value_counts().head(8))

In [ ]:
import numpy as np
from sklearn.metrics import f1_score
yt=te.iloc[:,41].astype(str).str.strip()
ytr=tr.iloc[:,41].astype(str).str.strip()
dos="back land neptune pod smurf teardrop apache2 udpstorm processtable worm mailbomb".split()
prb="satan ipsweep nmap portsweep mscan saint".split()
r2l="guess_passwd ftp_write imap phf multihop warezmaster warezclient spy xlock xsnoop snmpguess snmpgetattack sendmail named".split()
u2r="buffer_overflow loadmodule rootkit perl sqlattack xterm ps".split()
def fam(a,http_u2r):
    if a=="normal": return "Normal"
    if a in dos: return "DoS"
    if a in prb: return "Probe"
    if a in r2l: return "R2L"
    if a in u2r: return "U2R"
    if a=="httptunnel": return "U2R" if http_u2r else "R2L"
    return "UNMAPPED:"+a
for hv in [False,True]:
    print("httptunnel as U2R:",hv)
    print(yt.map(lambda a:fam(a,hv)).value_counts().to_dict())
print(yt[yt.isin(u2r+["httptunnel"])].value_counts().to_dict())

In [ ]:
D="/content/drive/MyDrive/phd_data/"
print(open(D+"nsl_final_test_results.csv").read()[:900])
kt=np.load(D+"nsl_test_p_x.npy")
cl=["DoS","Normal","Probe","R2L","U2R"]
for hv in [False,True]:
    y=yt.map(lambda a:fam(a,hv))
    ok=~y.str.startswith("UNMAPPED")
    yi=y[ok].map({c:i for i,c in enumerate(cl)}).values
    print(hv, kt.shape, round(f1_score(yi,kt[ok.values].argmax(1),average="macro"),4))

In [ ]:
cl=["DoS","Normal","Probe","R2L","U2R"]; N=1
ytf=yt.map(lambda a:fam(a,False))
yrf=ytr.map(lambda a:fam(a,False))
print("unmapped:", (ytf.str.startswith("UNM")).sum(), (yrf.str.startswith("UNM")).sum())
yi=ytf.map({c:i for i,c in enumerate(cl)}).values
yo=yrf.map({c:i for i,c in enumerate(cl)}).values
ko=np.load(D+"nsl_oof_xgb_s0.npy")
seen=set(ytr.unique()); unseen=(~yt.isin(seen)).values
print("unseen-type test rows:", int(unseen.sum()))
def ev2(y,P,w,un=None):
    Q=P.copy(); Q[:,N]*=w; p=Q.argmax(1)
    f1=f1_score(y,p,average="macro")
    fp=(p[y==N]!=N).mean()
    att=(y!=N)
    ds=(p[att&~un]!=N).mean() if un is not None else float("nan")
    du=(p[att&un]!=N).mean() if un is not None else float("nan")
    return round(f1,4),round(fp,4),round(ds,4),round(du,4)
ws=[0.1,0.25,0.5,0.75,1,1.5,2,4]
print("OOF  w, F1, FPR")
for w in ws: print(w, ev2(yo,ko,w)[:2])
print("TEST w, F1, FPR, det_seen, det_unseen")
for w in ws: print(w, ev2(yi,kt,w,unseen))
from sklearn.metrics import recall_score
for w in [0.25,1,2]:
    Q=kt.copy(); Q[:,N]*=w
    print(w, cl, np.round(recall_score(yi,Q.argmax(1),average=None,labels=range(5)),3))

In [ ]:
ws2=[0.01,0.02,0.05,0.1,0.25,0.5,1]
r=[(w,)+ev2(yo,ko,w)[:2] for w in ws2]
for x in r: print("OOF",x)
b=max(r,key=lambda x:x[1])[0]
print("chosen w:",b)
print("TEST", ev2(yi,kt,b,unseen))
print("TEST raw", ev2(yi,kt,1,unseen))

In [ ]:
from sklearn.linear_model import LogisticRegression
D="/content/drive/MyDrive/phd_data/"
kor=np.load(D+"nsl_oof_rf_s0.npy"); koc=np.load(D+"nsl_oof_cnn_s0.npy")
ktr=np.load(D+"nsl_test_p_r.npy");  ktc=np.load(D+"nsl_test_p_c.npy")
def stack(Xo,Xt):
    m=LogisticRegression(max_iter=3000).fit(Xo,yo)
    return m.predict_proba(Xo), m.predict_proba(Xt)
so2,st2=stack(np.hstack([ko,kor]),np.hstack([kt,ktr]))
so3,st3=stack(np.hstack([ko,kor,koc]),np.hstack([kt,ktr,ktc]))
ws=[0.01,0.02,0.05,0.1,0.25,0.5,1,2]
for name,po,pt_ in [("RF raw",kor,ktr),("Stack RF+XGB",so2,st2),("Stack RF+XGB+CNN",so3,st3)]:
    r=[(w,)+ev2(yo,po,w)[:2] for w in ws]
    b=max(r,key=lambda x:x[1])[0]
    print(name,"| chosen w on OOF:",b)
    print("  OOF w=1:",ev2(yo,po,1)[:2],"w=chosen:",ev2(yo,po,b)[:2])
    print("  TEST w=1:",ev2(yi,pt_,1,unseen))
    print("  TEST w=chosen:",ev2(yi,pt_,b,unseen))

In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
D="/content/drive/MyDrive/phd_data/"
ua=pd.read_csv(D+"UNSW_NB15_training-set (1).csv")
ub=pd.read_csv(D+"UNSW_NB15_testing-set (1).csv")
le=LabelEncoder().fit(ua["attack_cat"])
yU=le.transform(ua["attack_cat"]); yV=le.transform(ub["attack_cat"])
NU=int(le.transform(["Normal"])[0])
oxx=np.load(D+"oof_xgb.npy"); orf=np.load(D+"oof_rf.npy"); ocn=np.load(D+"oof_cnn.npy")
tx=np.load(D+"unsw_test_p_x_s0.npy"); tr_=np.load(D+"unsw_test_p_r_s0.npy"); tc=np.load(D+"unsw_test_p_c_s0.npy")
def stk(Xo,Xt):
    m=LogisticRegression(max_iter=3000).fit(Xo,yU)
    return m.predict_proba(Xt)
s2=stk(np.hstack([oxx,orf]),np.hstack([tx,tr_]))
s3=stk(np.hstack([oxx,orf,ocn]),np.hstack([tx,tr_,tc]))
def evU(P,w):
    Q=P.copy(); Q[:,NU]*=w; p=Q.argmax(1)
    f1=f1_score(yV,p,average="macro")
    fa=(p[yV==NU]!=NU).mean(); det=(p[yV!=NU]!=NU).mean()
    return round(f1,4),round(float(fa),4),round(float(det),4)
for name,P in [("XGB",tx),("RF",tr_),("Stack RF+XGB",s2),("Stack RF+XGB+CNN",s3)]:
    print(name,"(macro-F1, false alarm, attack detection)")
    for w in [1,2,4,8]: print("  w=%s"%w, evU(P,w))

In [ ]:
g=te.iloc[:,:41].astype(str).agg("|".join,axis=1).factorize()[0]
ng=g.max()+1
def pred(P,w):
    Q=P.copy(); Q[:,N]*=w; return Q.argmax(1)
def wf1(y,p,wt):
    cm=np.bincount(y*5+p,weights=wt,minlength=25).reshape(5,5)
    tp=np.diag(cm); pr=cm.sum(0); rc=cm.sum(1)
    f=np.where(pr+rc>0,2*tp/np.maximum(pr+rc,1e-12),0)
    return f.mean()
S={"XGB w=0.1":pred(kt,0.1),"XGB raw":pred(kt,1),
   "RF w=0.5":pred(ktr,0.5),"Stack+CNN w=0.02":pred(st3,0.02),
   "Stack+CNN raw":pred(st3,1)}
pairs=[("XGB w=0.1","Stack+CNN w=0.02"),("XGB w=0.1","RF w=0.5"),
       ("XGB w=0.1","XGB raw"),("XGB raw","Stack+CNN raw")]
rng=np.random.default_rng(0)
B=1000; D_={p:[] for p in pairs}
for _ in range(B):
    c=np.bincount(rng.integers(0,ng,ng),minlength=ng).astype(float)
    wt=c[g]
    for a,b in pairs:
        D_[(a,b)].append(wf1(yi,S[a],wt)-wf1(yi,S[b],wt))
for a,b in pairs:
    d=np.array(D_[(a,b)])
    print(a,"minus",b,"| diff %.4f  95%% CI %.4f to %.4f"%(wf1(yi,S[a],np.ones(len(yi)))-wf1(yi,S[b],np.ones(len(yi))),*np.percentile(d,[2.5,97.5])))

In [ ]:
import os, numpy as np, pandas as pd
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.metrics import f1_score
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier

D = "/content/drive/MyDrive/phd_data"
tr = pd.read_csv(f"{D}/UNSW_NB15_training-set (1).csv")
feats = [c for c in tr.columns if c not in ("id", "attack_cat", "label")]
cat_cols = ["proto", "service", "state"]
le = LabelEncoder().fit(tr["attack_cat"].str.strip())
ytr = le.transform(tr["attack_cat"].str.strip())
K = len(le.classes_)

Xtr = tr[feats].copy()
Xtr[cat_cols] = OrdinalEncoder(handle_unknown="use_encoded_value",
                               unknown_value=-1).fit_transform(Xtr[cat_cols].astype(str))

fold_id = np.load(f"{D}/fold_id.npy")
print("rows", len(tr), "| classes", K, "| folds", np.bincount(fold_id))

# keep the old 200-tree files
for f in ["oof_xgb.npy", "oof_rf.npy"]:
    if os.path.exists(f"{D}/{f}") and not os.path.exists(f"{D}/old200_{f}"):
        os.rename(f"{D}/{f}", f"{D}/old200_{f}")

oxx = np.zeros((len(tr), K)); orf = np.zeros((len(tr), K))
for k in range(10):
    a = np.where(fold_id != k)[0]; b = np.where(fold_id == k)[0]
    m = XGBClassifier(objective="multi:softprob", n_estimators=300, learning_rate=0.1,
                      max_depth=8, tree_method="hist", n_jobs=-1, random_state=0)
    m.fit(Xtr.iloc[a], ytr[a]); oxx[b] = m.predict_proba(Xtr.iloc[b])
    r = RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=0)
    r.fit(Xtr.iloc[a], ytr[a]); orf[b] = r.predict_proba(Xtr.iloc[b])
    print("fold", k, "done")

np.save(f"{D}/oof_xgb.npy", oxx); np.save(f"{D}/oof_rf.npy", orf)
print("XGB OOF macro-F1:", round(f1_score(ytr, oxx.argmax(1), average="macro"), 4))
print("RF  OOF macro-F1:", round(f1_score(ytr, orf.argmax(1), average="macro"), 4))

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

n = list(le.classes_).index("Normal")
ocn = np.load(f"{D}/oof_cnn.npy")
old_x = np.load(f"{D}/old200_oof_xgb.npy"); old_r = np.load(f"{D}/old200_oof_rf.npy")

def cv_stack(parts):
    Z = np.hstack(parts); out = np.zeros((len(Z), K))
    for k in range(10):
        a, b = fold_id != k, fold_id == k
        lr = LogisticRegression(max_iter=2000).fit(Z[a], ytr[a])
        out[b] = lr.predict_proba(Z[b])
    return out

def row(name, p):
    pr = p.argmax(1)
    return (name, round(f1_score(ytr, pr, average="macro"), 4),
            round((pr[ytr == n] != n).mean(), 4), round((pr == ytr).mean(), 4))

rows = []
for tag, ox_, or_ in [("OLD200", old_x, old_r), ("NEW300", oxx, orf)]:
    rows += [row(f"{tag} XGBoost", ox_), row(f"{tag} RF", or_),
             row(f"{tag} LR on XGB", cv_stack([ox_])),
             row(f"{tag} Stack RF+XGB", cv_stack([or_, ox_])),
             row(f"{tag} Stack RF+XGB+CNN", cv_stack([or_, ox_, ocn]))]
rows.append(row("CNN (unchanged)", ocn))
print(f"{'system':28s} macroF1   FPR    acc")
for r in rows: print(f"{r[0]:28s} {r[1]:.4f}  {r[2]:.4f}  {r[3]:.4f}")

In [ ]:
import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

te = pd.read_csv(f"{D}/UNSW_NB15_testing-set (1).csv")
yte = le.transform(te["attack_cat"].str.strip())
n = list(le.classes_).index("Normal")

def fit_meta(oofs):
    return LogisticRegression(max_iter=2000).fit(np.hstack(oofs), ytr)

M = {"LR on XGB": (fit_meta([oxx]), ["x"]),
     "Stack RF+XGB": (fit_meta([orf, oxx]), ["r", "x"]),
     "Stack RF+XGB+CNN": (fit_meta([orf, oxx, ocn]), ["r", "x", "c"])}

def score(P, w=1.0):
    P = P.copy(); P[:, n] *= w
    pr = P.argmax(1); att = yte != n
    return (f1_score(yte, pr, average="macro"),
            (pr[~att] != n).mean(),            # false alarm rate
            (pr[att] != n).mean())             # attack detection

res = []
for s in [0, 1, 2]:
    pt = {k: np.load(f"{D}/unsw_test_p_{k}_s{s}.npy") for k in "xrc"}
    systems = {"XGBoost raw": pt["x"], "RF raw": pt["r"], "CNN raw": pt["c"]}
    for name, (m, parts) in M.items():
        systems[name] = m.predict_proba(np.hstack([pt[p] for p in parts]))
    for name, P in systems.items():
        for w in [1, 1.25, 2, 4, 8]:
            f, fpr, det = score(P, w)
            res.append((s, name, w, f, fpr, det))

df = pd.DataFrame(res, columns=["seed", "system", "w", "F1", "FPR", "det"])
g = df.groupby(["system", "w"])[["F1", "FPR", "det"]].agg(["mean", "std"]).round(4)
print("=== w = 1 (raw argmax), mean and std over 3 seeds ===")
print(g.xs(1, level="w").to_string())
print("\n=== threshold sweep, 3-seed mean ===")
print(df.groupby(["system", "w"])[["F1", "FPR", "det"]].mean().round(4).to_string())

In [ ]:
import numpy as np, pandas as pd
from itertools import combinations
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.multitest import multipletests

pt = {k: np.load(f"{D}/unsw_test_p_{k}_s0.npy") for k in "xrc"}   # seed 0
def meta_p(m, parts): return m.predict_proba(np.hstack([pt[p] for p in parts]))
P = {"XGB raw": pt["x"],
     "LR on XGB": meta_p(*M["LR on XGB"]),
     "Stack": meta_p(*M["Stack RF+XGB"]),
     "Stack+CNN": meta_p(*M["Stack RF+XGB+CNN"])}
def wpred(Pm, w): Q = Pm.copy(); Q[:, n] *= w; return Q.argmax(1)
pred = {k: v.argmax(1) for k, v in P.items()}
pred["XGB w=4"] = wpred(P["XGB raw"], 4)

# rows of test whose features also appear in train
feats_ = [c for c in tr.columns if c not in ("id", "attack_cat", "label")]
h_tr = pd.util.hash_pandas_object(tr[feats_], index=False)
h_te = pd.util.hash_pandas_object(te[feats_], index=False)
nonov = ~h_te.isin(set(h_tr)).values
print("non-overlapping test rows:", nonov.sum())
print("macro-F1 on non-overlap rows:")
for k, v in pred.items():
    print(f"  {k:10s} {f1_score(yte[nonov], v[nonov], average='macro'):.4f}")

# McNemar + Holm
pairs = [("LR on XGB", "XGB raw"), ("Stack", "XGB raw"), ("Stack+CNN", "XGB raw"),
         ("Stack", "LR on XGB"), ("Stack+CNN", "Stack"), ("Stack+CNN", "XGB w=4")]
pv = []
for a_, b_ in pairs:
    ca, cb = pred[a_] == yte, pred[b_] == yte
    t = [[int((ca & cb).sum()), int((ca & ~cb).sum())],
         [int((~ca & cb).sum()), int((~ca & ~cb).sum())]]
    pv.append(mcnemar(t, exact=False, correction=True).pvalue)
hp = multipletests(pv, method="holm")[1]
print("\nMcNemar (Holm-adjusted p):")
for (a_, b_), p_ in zip(pairs, hp): print(f"  {a_} vs {b_}: p = {p_:.2e}")

# grouped bootstrap (groups = identical feature rows in test), 1000 resamples
gid = te.groupby(feats_, dropna=False).ngroup().values
G = gid.max() + 1
rng = np.random.default_rng(0)
def mf1(y, pr, rw):
    C = np.bincount(y * K + pr, weights=rw, minlength=K * K).reshape(K, K)
    tp = np.diag(C); fp = C.sum(0) - tp; fn = C.sum(1) - tp
    d = 2 * tp + fp + fn
    return np.where(d > 0, 2 * tp / np.where(d > 0, d, 1), 0).mean()
boot = {p: [] for p in pairs}
for _ in range(1000):
    rw = np.bincount(rng.integers(0, G, G), minlength=G)[gid].astype(float)
    sc = {k: mf1(yte, v, rw) for k, v in pred.items()}
    for a_, b_ in pairs: boot[(a_, b_)].append(sc[a_] - sc[b_])
print("\nBootstrap difference in macro-F1 (95% CI):")
for (a_, b_), v in boot.items():
    lo, hi = np.percentile(v, [2.5, 97.5])
    print(f"  {a_} - {b_}: {np.mean(v):+.4f}  [{lo:+.4f}, {hi:+.4f}]")

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

def cv_stack(parts):
    Z = np.hstack(parts); out = np.zeros((len(Z), K))
    for k in range(10):
        a, b = fold_id != k, fold_id == k
        out[b] = LogisticRegression(max_iter=2000).fit(Z[a], ytr[a]).predict_proba(Z[b])
    return out

pr = {"XGB": oxx.argmax(1),
      "LR on XGB": cv_stack([oxx]).argmax(1),
      "Stack": cv_stack([orf, oxx]).argmax(1),
      "Stack+CNN": cv_stack([orf, oxx, ocn]).argmax(1)}

# groups = identical feature rows in train
feats_ = [c for c in tr.columns if c not in ("id", "attack_cat", "label")]
gid = tr.groupby(feats_, dropna=False).ngroup().values
G = gid.max() + 1
rng = np.random.default_rng(0)

def mf1(y, p, rw):
    C = np.bincount(y * K + p, weights=rw, minlength=K * K).reshape(K, K)
    tp = np.diag(C); fp = C.sum(0) - tp; fn = C.sum(1) - tp
    d = 2 * tp + fp + fn
    return np.where(d > 0, 2 * tp / np.where(d > 0, d, 1), 0).mean()

pairs = [("Stack", "LR on XGB"), ("Stack+CNN", "Stack"),
         ("Stack", "XGB"), ("LR on XGB", "XGB")]
boot = {p: [] for p in pairs}
for _ in range(1000):
    rw = np.bincount(rng.integers(0, G, G), minlength=G)[gid].astype(float)
    sc = {k: mf1(ytr, v, rw) for k, v in pr.items()}
    for a_, b_ in pairs: boot[(a_, b_)].append(sc[a_] - sc[b_])
print("UNSW CV bootstrap, difference in macro-F1 (95% CI):")
for (a_, b_), v in boot.items():
    lo, hi = np.percentile(v, [2.5, 97.5])
    print(f"  {a_} - {b_}: {np.mean(v):+.4f}  [{lo:+.4f}, {hi:+.4f}]")

In [ ]:
import numpy as np, pandas as pd
from sklearn.metrics import precision_recall_fscore_support as prfs

n = list(le.classes_).index("Normal")

def perclass(pred):
    p, r, f, s = prfs(yte, pred, labels=range(K), zero_division=0)
    return pd.DataFrame({"F1": f, "recall": r, "support": s}, index=le.classes_)

def stack_p(seed, parts, m):
    pt = {k: np.load(f"{D}/unsw_test_p_{k}_s{seed}.npy") for k in "xrc"}
    return m.predict_proba(np.hstack([pt[p] for p in parts]))

rows = {}
for s in [0, 1, 2]:
    px = np.load(f"{D}/unsw_test_p_x_s{s}.npy")
    Q = px.copy(); Q[:, n] *= 4
    ms, parts_s = M["Stack RF+XGB"]
    mc, parts_c = M["Stack RF+XGB+CNN"]
    for name, pred in {"XGB raw": px.argmax(1), "XGB w=4": Q.argmax(1),
                       "Stack": stack_p(s, parts_s, ms).argmax(1),
                       "Stack+CNN": stack_p(s, parts_c, mc).argmax(1)}.items():
        rows.setdefault(name, []).append(perclass(pred))

f1 = pd.DataFrame({k: sum(d["F1"] for d in v) / 3 for k, v in rows.items()}).round(3)
rc = pd.DataFrame({k: sum(d["recall"] for d in v) / 3 for k, v in rows.items()}).round(3)
f1["support"] = rows["XGB raw"][0]["support"].astype(int)
print("=== UNSW test, per-class F1 (3-seed mean) ===");   print(f1.to_string())
print("\n=== UNSW test, per-class recall (3-seed mean) ===");  print(rc.to_string())

In [ ]:
import numpy as np, pandas as pd
from sklearn.metrics import precision_recall_fscore_support as prfs

OUT = "/content/drive/MyDrive/phd_data"
names = ["DoS", "Normal", "Probe", "R2L", "U2R"]          # Normal = index 1
px = np.load(f"{OUT}/nsl_test_p_x.npy")

# pick whichever label vector has the test length
ytest = next(v for v in (yi, yo) if len(v) == len(px))
print("test rows:", len(px), "| label vector length:", len(ytest),
      "| counts:", dict(zip(names, np.bincount(ytest, minlength=5))))

Q = px.copy(); Q[:, 1] *= 0.1
res = {}
for k, pr_ in {"raw": px.argmax(1), "w=0.1": Q.argmax(1)}.items():
    p, r, f, s = prfs(ytest, pr_, labels=range(5), zero_division=0)
    res[k] = (f, r, s)
tab = pd.DataFrame({"support": res["raw"][2],
                    "F1 raw": res["raw"][0], "recall raw": res["raw"][1],
                    "F1 w=0.1": res["w=0.1"][0], "recall w=0.1": res["w=0.1"][1]},
                   index=names).round(3)
print(tab.to_string())

# appendix: every test attack name -> family, with counts
lab = te.iloc[:, 41].astype(str).str.strip().str.rstrip(".")
fam_of = {a: "DoS" for a in dos} | {a: "Probe" for a in prb} | \
         {a: "R2L" for a in r2l} | {a: "U2R" for a in u2r}
app = (lab.value_counts().rename("test rows").to_frame()
         .assign(family=lambda d: [fam_of.get(i, "Normal" if i == "normal" else "UNMAPPED")
                                   for i in d.index]))
print(app.sort_values(["family", "test rows"], ascending=[True, False]).to_string())

In [ ]:
import pandas as pd, numpy as np

P = "/content/drive/MyDrive/phd_data/KDDTest_Labelled.txt"
kte = pd.read_csv(P, header=None, sep=",")
print("shape:", kte.shape)                        # expect (22544, 43)
lab = kte.iloc[:, 41].astype(str).str.strip().str.rstrip(".")
print("rows:", len(lab), "| distinct labels:", lab.nunique())

fam_of = {a: "DoS" for a in dos} | {a: "Probe" for a in prb} | \
         {a: "R2L" for a in r2l} | {a: "U2R" for a in u2r}
app = (lab.value_counts().rename("test rows").to_frame()
         .assign(family=lambda d: [fam_of.get(i, "Normal" if i == "normal" else "UNMAPPED")
                                   for i in d.index]))
print(app.sort_values(["family", "test rows"], ascending=[True, False]).to_string())
print("\nfamily totals:")
print(app.groupby("family")["test rows"].sum())

In [ ]:
import numpy as np, pandas as pd
from sklearn.metrics import precision_recall_fscore_support as prfs, f1_score

names = ["DoS", "Normal", "Probe", "R2L", "U2R"]
OUT = "/content/drive/MyDrive/phd_data"
px = np.load(f"{OUT}/nsl_test_p_x.npy")

# kte and lab come from the previous cell (KDDTest_Labelled.txt)
mask = (lab == "httptunnel").values
print("httptunnel rows:", mask.sum(), "| current labels on those rows:",
      np.bincount(yi[mask], minlength=5))        # expect all in R2L = index 3

yi_std = yi.copy(); yi_std[mask] = 4              # standard mapping: httptunnel -> U2R
print("counts, standard mapping:", dict(zip(names, np.bincount(yi_std, minlength=5))))

Q = px.copy(); Q[:, 1] *= 0.1
rows = []
for tag, y in [("R2L mapping (old)", yi), ("U2R mapping (standard)", yi_std)]:
    for setting, pr_ in [("raw", px.argmax(1)), ("w=0.1", Q.argmax(1))]:
        rows.append((tag, setting, round(f1_score(y, pr_, average="macro"), 4)))
print(pd.DataFrame(rows, columns=["mapping", "setting", "macro-F1"]).to_string(index=False))

for setting, pr_ in [("raw", px.argmax(1)), ("w=0.1", Q.argmax(1))]:
    p, r, f, s = prfs(yi_std, pr_, labels=range(5), zero_division=0)
    print(f"\nStandard mapping, XGBoost {setting}:")
    print(pd.DataFrame({"support": s, "F1": f, "recall": r}, index=names).round(3).to_string())

In [ ]:
import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.multitest import multipletests

OUT = "/content/drive/MyDrive/phd_data"
ld = lambda f: np.load(f"{OUT}/{f}")
oof = {"x": ld("nsl_oof_xgb_s0.npy"), "r": ld("nsl_oof_rf_s0.npy"), "c": ld("nsl_oof_cnn_s0.npy")}
tst = {"x": ld("nsl_test_p_x.npy"), "r": ld("nsl_test_p_r.npy"), "c": ld("nsl_test_p_c.npy")}
ytest = yi_std                       # standard mapping (httptunnel -> U2R)
K, N = 5, 1                          # Normal = index 1

def meta(parts):
    m = LogisticRegression(max_iter=2000).fit(np.hstack([oof[p] for p in parts]), yo)
    return (m.predict_proba(np.hstack([oof[p] for p in parts])),
            m.predict_proba(np.hstack([tst[p] for p in parts])))

S = {"XGBoost": (oof["x"], tst["x"]), "RF": (oof["r"], tst["r"]), "CNN": (oof["c"], tst["c"]),
     "LR on XGBoost": meta(["x"]), "Stack RF+XGB": meta(["r", "x"]),
     "Stack RF+XGB+CNN": meta(["r", "x", "c"])}

grid = [0.01, 0.02, 0.05, 0.1, 0.2, 0.5, 1.0]
def pred(P, w): Q = P.copy(); Q[:, N] *= w; return Q.argmax(1)
def fpr(y, p): return (p[y == N] != N).mean()

chosen, rows = {}, []
for name, (po, pt) in S.items():
    w_star = max(grid, key=lambda w: f1_score(yo, pred(po, w), average="macro"))
    chosen[name] = w_star
    for tag, w in [("raw", 1.0), (f"chosen w={w_star}", w_star), ("w=0.1", 0.1)]:
        p = pred(pt, w)
        rows.append((name, tag, round(f1_score(ytest, p, average="macro"), 4), round(fpr(ytest, p), 4)))
print(pd.DataFrame(rows, columns=["system", "setting", "macro-F1", "FPR"]).to_string(index=False))

# systems used in the comparisons
test_pred = {"XGB raw": pred(S["XGBoost"][1], 1), "XGB w=0.1": pred(S["XGBoost"][1], 0.1),
             "LR on XGB raw": pred(S["LR on XGBoost"][1], 1),
             "RF chosen": pred(S["RF"][1], chosen["RF"]),
             "Stack+CNN raw": pred(S["Stack RF+XGB+CNN"][1], 1),
             "Stack+CNN chosen": pred(S["Stack RF+XGB+CNN"][1], chosen["Stack RF+XGB+CNN"])}
pairs = [("XGB w=0.1", "XGB raw"), ("XGB w=0.1", "Stack+CNN chosen"), ("XGB w=0.1", "RF chosen"),
         ("XGB raw", "Stack+CNN raw"), ("XGB raw", "LR on XGB raw")]

# McNemar + Holm
pv = []
for a_, b_ in pairs:
    ca, cb = test_pred[a_] == ytest, test_pred[b_] == ytest
    pv.append(mcnemar([[int((ca & cb).sum()), int((ca & ~cb).sum())],
                       [int((~ca & cb).sum()), int((~ca & ~cb).sum())]],
                      exact=False, correction=True).pvalue)
print("\nMcNemar, Holm-adjusted p:")
for (a_, b_), p_ in zip(pairs, multipletests(pv, method="holm")[1]): print(f"  {a_} vs {b_}: {p_:.3g}")

# grouped bootstrap (identical test rows resampled together)
gid = kte.iloc[:, :41].astype(str).agg("|".join, axis=1).factorize()[0]
G = gid.max() + 1; rng = np.random.default_rng(0)
def mf1(y, p, rw):
    C = np.bincount(y * K + p, weights=rw, minlength=K * K).reshape(K, K)
    tp = np.diag(C); fp = C.sum(0) - tp; fn = C.sum(1) - tp; d = 2 * tp + fp + fn
    return np.where(d > 0, 2 * tp / np.where(d > 0, d, 1), 0).mean()
boot = {p: [] for p in pairs}
for _ in range(1000):
    rw = np.bincount(rng.integers(0, G, G), minlength=G)[gid].astype(float)
    sc = {k: mf1(ytest, v, rw) for k, v in test_pred.items()}
    for a_, b_ in pairs: boot[(a_, b_)].append(sc[a_] - sc[b_])
print("\nBootstrap, macro-F1 difference (95% CI):")
for (a_, b_), v in boot.items():
    lo, hi = np.percentile(v, [2.5, 97.5]); print(f"  {a_} - {b_}: {np.mean(v):+.4f} [{lo:+.4f}, {hi:+.4f}]")
print("\nchosen weights on OOF:", chosen)

In [ ]:
import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

OUT = "/content/drive/MyDrive/phd_data"
ld = lambda f: np.load(f"{OUT}/{f}")
oof = {"x": ld("nsl_oof_xgb_s0.npy"), "r": ld("nsl_oof_rf_s0.npy"), "c": ld("nsl_oof_cnn_s0.npy")}
def fit(parts): return LogisticRegression(max_iter=2000).fit(np.hstack([oof[p] for p in parts]), yo)
M2 = {"LR on XGB": (fit(["x"]), ["x"]), "Stack RF+XGB": (fit(["r", "x"]), ["r", "x"]),
      "Stack RF+XGB+CNN": (fit(["r", "x", "c"]), ["r", "x", "c"])}
rows = []
for s, suf in [(0, ""), (1, "_s1"), (2, "_s2")]:
    t = {k: ld(f"nsl_test_p_{k}{suf}.npy") for k in "xrc"}
    S = {"XGBoost raw": t["x"], "RF raw": t["r"], "CNN raw": t["c"]}
    for name, (m, parts) in M2.items():
        S[name] = m.predict_proba(np.hstack([t[p] for p in parts]))
    for name, P in S.items():
        rows.append((s, name, f1_score(yi_std, P.argmax(1), average="macro")))
d = pd.DataFrame(rows, columns=["seed", "system", "F1"])
print(d.pivot(index="system", columns="seed", values="F1").round(4))
print(d.groupby("system")["F1"].agg(["mean", "std"]).round(4))